# Qwen3-0.6B 텔레그램 액션 파서 — T4 학습 → 검증 → GGUF 양자화

위에서부터 순서대로 실행하면 Google Drive 마운트, 데이터 확인, LoRA 학습 및 체크포인트 저장, test 검증, LoRA 병합, llama.cpp 빌드, F16 GGUF 및 Q4/Q5/Q8 양자화를 수행합니다.

**Colab 설정:** `런타임 → 런타임 유형 변경 → T4 GPU`

기본 데이터 위치는 다음 중 하나입니다.

- `/content/drive/MyDrive/telegram_finetune_retrain/`
- `/content/drive/MyDrive/telegram_finetune_retrain.zip`

모든 지속 저장 경로와 캐시는 `/content/drive/MyDrive` 아래를 사용합니다.


In [ ]:
# 1. Google Drive 연결 및 모든 저장 경로 설정
from google.colab import drive
drive.mount("/content/drive")

import os
from pathlib import Path

DRIVE_ROOT = Path("/content/drive/MyDrive")
PROJECT_ROOT = DRIVE_ROOT / "qwen3_telegram_pipeline"

DATA_DIR = DRIVE_ROOT / "telegram_finetune_retrain_260720"
DATA_ZIP = DRIVE_ROOT / "telegram_finetune_retrain_260720.zip"

BASE_MODEL_DIR = PROJECT_ROOT / "base_model" / "Qwen3-0.6B"
CACHE_ROOT = PROJECT_ROOT / "cache"
HF_HOME = CACHE_ROOT / "huggingface"
HF_DATASETS_CACHE = CACHE_ROOT / "datasets"
TMP_DIR = PROJECT_ROOT / "tmp"

RUN_NAME = "qwen3_0.6b_telegram_parser_260720"
RUN_ROOT = PROJECT_ROOT / "runs" / RUN_NAME
CHECKPOINT_DIR = RUN_ROOT / "checkpoints"
ADAPTER_DIR = RUN_ROOT / "lora_adapter"
MERGED_DIR = RUN_ROOT / "merged_model"
EVAL_DIR = RUN_ROOT / "evaluation"
GGUF_DIR = RUN_ROOT / "gguf"
LLAMA_CPP_DIR = PROJECT_ROOT / "tools" / "llama.cpp"

for path in [
    PROJECT_ROOT, DATA_DIR, BASE_MODEL_DIR.parent, CACHE_ROOT,
    HF_HOME, HF_DATASETS_CACHE, TMP_DIR, RUN_ROOT,
    CHECKPOINT_DIR, ADAPTER_DIR, MERGED_DIR, EVAL_DIR,
    GGUF_DIR, LLAMA_CPP_DIR.parent,
]:
    path.mkdir(parents=True, exist_ok=True)

os.environ["HF_HOME"] = str(HF_HOME)
os.environ["HF_HUB_CACHE"] = str(HF_HOME / "hub")
os.environ["TRANSFORMERS_CACHE"] = str(HF_HOME / "hub")
os.environ["HF_DATASETS_CACHE"] = str(HF_DATASETS_CACHE)
os.environ["TORCH_HOME"] = str(CACHE_ROOT / "torch")
os.environ["TMPDIR"] = str(TMP_DIR)
os.environ["TOKENIZERS_PARALLELISM"] = "false"

print("프로젝트 루트:", PROJECT_ROOT)
print("실행 결과 루트:", RUN_ROOT)


Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
프로젝트 루트: /content/drive/MyDrive/qwen3_telegram_pipeline
실행 결과 루트: /content/drive/MyDrive/qwen3_telegram_pipeline/runs/qwen3_0.6b_telegram_parser_260720


In [ ]:
# 2. 라이브러리 설치
import subprocess
import sys

subprocess.run([sys.executable, "-m", "pip", "uninstall", "-y", "torchao"], check=False)

packages = [
    "transformers==4.51.3",
    "tokenizers==0.21.1",
    "datasets==3.5.0",
    "peft==0.15.2",
    "accelerate==1.6.0",
    "huggingface_hub==0.30.2",
    "safetensors>=0.5.3",
    "sentencepiece",
    "protobuf",
    "psutil",
]
subprocess.run([sys.executable, "-m", "pip", "install", "-q", "-U", *packages], check=True)
print("라이브러리 설치 완료")


라이브러리 설치 완료


In [ ]:
# 3. 환경 및 GPU 확인
import platform
import torch
import transformers
import datasets
import peft
import accelerate

print("Python:", platform.python_version())
print("PyTorch:", torch.__version__)
print("Transformers:", transformers.__version__)
print("Datasets:", datasets.__version__)
print("PEFT:", peft.__version__)
print("Accelerate:", accelerate.__version__)
print("CUDA 사용 가능:", torch.cuda.is_available())

if not torch.cuda.is_available():
    raise RuntimeError("GPU 런타임이 아닙니다. Colab에서 T4 GPU로 변경하세요.")

gpu_name = torch.cuda.get_device_name(0)
print("GPU:", gpu_name)
print("BF16 지원:", torch.cuda.is_bf16_supported())


Python: 3.12.13
PyTorch: 2.11.0+cu128
Transformers: 4.51.3
Datasets: 3.5.0
PEFT: 0.15.2
Accelerate: 1.6.0
CUDA 사용 가능: True
GPU: Tesla T4
BF16 지원: True


In [ ]:
# 4. 데이터 폴더 준비 및 파일 확인
import shutil
import zipfile

required_names = ["train.jsonl", "validation.jsonl", "test.jsonl"]

def has_required_files(directory: Path) -> bool:
    return all((directory / name).exists() for name in required_names)

if not has_required_files(DATA_DIR):
    if not DATA_ZIP.exists():
        raise FileNotFoundError(
            f"{DATA_DIR} 또는 {DATA_ZIP}에 재학습 데이터를 넣어주세요."
        )

    print("ZIP 압축 해제:", DATA_ZIP)
    with zipfile.ZipFile(DATA_ZIP, "r") as zf:
        zf.extractall(DATA_DIR)

    if not has_required_files(DATA_DIR):
        candidates = [
            p for p in DATA_DIR.rglob("train.jsonl")
            if (p.parent / "validation.jsonl").exists()
            and (p.parent / "test.jsonl").exists()
        ]
        if len(candidates) != 1:
            raise RuntimeError(f"데이터 위치를 확정할 수 없습니다: {candidates}")
        source_dir = candidates[0].parent
        for name in required_names:
            shutil.copy2(source_dir / name, DATA_DIR / name)

TRAIN_FILE = DATA_DIR / "train.jsonl"
VALID_FILE = DATA_DIR / "validation.jsonl"
TEST_FILE = DATA_DIR / "test.jsonl"

for path in [TRAIN_FILE, VALID_FILE, TEST_FILE]:
    count = sum(1 for line in path.open("r", encoding="utf-8") if line.strip())
    print(f"{path.name}: {count}행 — {path}")

expected_counts = {"train.jsonl":1604, "validation.jsonl":324, "test.jsonl":325}
for path in [TRAIN_FILE, VALID_FILE, TEST_FILE]:
    actual = sum(1 for line in path.open("r", encoding="utf-8") if line.strip())
    if actual != expected_counts[path.name]:
        print(f"경고: {path.name} 예상 {expected_counts[path.name]}행, 실제 {actual}행")


train.jsonl: 1624행 — /content/drive/MyDrive/telegram_finetune_retrain_260720/train.jsonl
validation.jsonl: 336행 — /content/drive/MyDrive/telegram_finetune_retrain_260720/validation.jsonl
test.jsonl: 325행 — /content/drive/MyDrive/telegram_finetune_retrain_260720/test.jsonl
경고: train.jsonl 예상 1604행, 실제 1624행
경고: validation.jsonl 예상 324행, 실제 336행


In [ ]:
# 5. JSONL 스키마 검증
import json

ALLOWED_ACTIONS = {
    "OPEN_LONG","OPEN_SHORT","ADD","SET_STOP","SET_TP",
    "CLOSE_HALF","CLOSE_ADDS","CLOSE_ALL","CANCEL_ADD","CANCEL_STOP",
}

def read_jsonl(path: Path):
    rows = []
    with path.open("r", encoding="utf-8") as file:
        for line_number, line in enumerate(file, 1):
            if not line.strip():
                continue
            try:
                rows.append(json.loads(line))
            except json.JSONDecodeError as exc:
                raise ValueError(f"{path.name} {line_number}번째 줄 JSON 오류: {exc}") from exc
    return rows

def validate_row(row, split_name, index):
    messages = row.get("messages")
    if not isinstance(messages, list) or len(messages) != 3:
        raise ValueError(f"{split_name}[{index}] messages 오류")
    roles = [message.get("role") for message in messages]
    if roles != ["system", "user", "assistant"]:
        raise ValueError(f"{split_name}[{index}] role 순서 오류: {roles}")

    label = json.loads(messages[2]["content"])
    if set(label.keys()) != {"actions"} or not isinstance(label["actions"], list):
        raise ValueError(f"{split_name}[{index}] actions 스키마 오류")

    for action in label["actions"]:
        if set(action.keys()) != {"type", "price"}:
            raise ValueError(f"{split_name}[{index}] action 키 오류: {action}")
        if action["type"] not in ALLOWED_ACTIONS:
            raise ValueError(f"{split_name}[{index}] action type 오류: {action}")
        if action["price"] is not None and not isinstance(action["price"], (int, float)):
            raise ValueError(f"{split_name}[{index}] price 타입 오류: {action}")

raw_rows = {
    "train": read_jsonl(TRAIN_FILE),
    "validation": read_jsonl(VALID_FILE),
    "test": read_jsonl(TEST_FILE),
}

for split_name, rows in raw_rows.items():
    for index, row in enumerate(rows):
        validate_row(row, split_name, index)
    print(f"{split_name}: {len(rows)}행 검증 완료")


train: 1624행 검증 완료
validation: 336행 검증 완료
test: 325행 검증 완료


In [ ]:
# 6. 베이스 모델을 Drive에 영구 다운로드
from huggingface_hub import snapshot_download

MODEL_ID = "Qwen/Qwen3-0.6B"

if not (BASE_MODEL_DIR / "model.safetensors").exists():
    print("베이스 모델 다운로드 시작:", MODEL_ID)
    snapshot_download(repo_id=MODEL_ID, local_dir=str(BASE_MODEL_DIR), max_workers=4)
    print("베이스 모델 다운로드 완료")
else:
    print("Drive의 기존 베이스 모델 사용:", BASE_MODEL_DIR)

required_base_files = [
    BASE_MODEL_DIR / "model.safetensors",
    BASE_MODEL_DIR / "config.json",
    BASE_MODEL_DIR / "tokenizer.json",
    BASE_MODEL_DIR / "tokenizer_config.json",
]
missing = [str(path) for path in required_base_files if not path.exists()]
if missing:
    raise FileNotFoundError(f"베이스 모델 파일 불완전: {missing}")


Drive의 기존 베이스 모델 사용: /content/drive/MyDrive/qwen3_telegram_pipeline/base_model/Qwen3-0.6B


In [ ]:
# 7. 토크나이저 로드 및 assistant 정답만 loss에 포함하는 전처리
from datasets import Dataset, DatasetDict
from transformers import AutoTokenizer

MAX_LENGTH = 256

tokenizer = AutoTokenizer.from_pretrained(str(BASE_MODEL_DIR), use_fast=True)
if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token
tokenizer.padding_side = "right"

def tokenize_row(example):
    messages = example["messages"]
    prompt_ids = tokenizer.apply_chat_template(
        messages[:2],
        tokenize=True,
        add_generation_prompt=True,
        enable_thinking=False,
    )
    completion_ids = tokenizer(
        messages[2]["content"] + tokenizer.eos_token,
        add_special_tokens=False,
    )["input_ids"]

    input_ids = prompt_ids + completion_ids
    labels = [-100] * len(prompt_ids) + completion_ids.copy()
    attention_mask = [1] * len(input_ids)

    if len(input_ids) > MAX_LENGTH:
        overflow = len(input_ids) - MAX_LENGTH
        prompt_cut = min(overflow, len(prompt_ids))
        input_ids = input_ids[prompt_cut:]
        labels = labels[prompt_cut:]
        attention_mask = attention_mask[prompt_cut:]

    if len(input_ids) > MAX_LENGTH:
        input_ids = input_ids[-MAX_LENGTH:]
        labels = labels[-MAX_LENGTH:]
        attention_mask = attention_mask[-MAX_LENGTH:]

    if not any(label != -100 for label in labels):
        raise ValueError("정답 토큰이 전부 잘렸습니다.")

    return {
        "input_ids": input_ids,
        "attention_mask": attention_mask,
        "labels": labels,
        "length": len(input_ids),
    }

dataset = DatasetDict({
    split: Dataset.from_list(rows)
    for split, rows in raw_rows.items()
})

tokenized = dataset.map(
    tokenize_row,
    remove_columns=["messages"],
    num_proc=1,
    desc="토큰화",
)

print(tokenized)
for split_name in tokenized:
    lengths = tokenized[split_name]["length"]
    print(
        split_name,
        "rows=", len(lengths),
        "min=", min(lengths),
        "max=", max(lengths),
        "avg=", round(sum(lengths) / len(lengths), 2),
    )


토큰화:   0%|          | 0/1624 [00:00<?, ? examples/s]

토큰화:   0%|          | 0/336 [00:00<?, ? examples/s]

토큰화:   0%|          | 0/325 [00:00<?, ? examples/s]

DatasetDict({
    train: Dataset({
        features: ['input_ids', 'attention_mask', 'labels', 'length'],
        num_rows: 1624
    })
    validation: Dataset({
        features: ['input_ids', 'attention_mask', 'labels', 'length'],
        num_rows: 336
    })
    test: Dataset({
        features: ['input_ids', 'attention_mask', 'labels', 'length'],
        num_rows: 325
    })
})
train rows= 1624 min= 90 max= 256 avg= 106.94
validation rows= 336 min= 91 max= 164 avg= 107.28
test rows= 325 min= 90 max= 164 avg= 106.41


In [ ]:
# 8. 학습 설정
FORCE_RETRAIN = False

NUM_EPOCHS = 5
TRAIN_BATCH_SIZE = 4
EVAL_BATCH_SIZE = 8
GRADIENT_ACCUMULATION_STEPS = 4
LEARNING_RATE = 1e-4
SAVE_STEPS = 50

if FORCE_RETRAIN:
    import shutil
    for path in [CHECKPOINT_DIR, ADAPTER_DIR, MERGED_DIR, EVAL_DIR, GGUF_DIR]:
        if path.exists():
            shutil.rmtree(path)
        path.mkdir(parents=True, exist_ok=True)
    print("기존 산출물을 삭제하고 새 학습 시작")
else:
    print("기존 adapter/checkpoint가 있으면 재사용 또는 이어서 학습")


기존 adapter/checkpoint가 있으면 재사용 또는 이어서 학습


In [ ]:
# 9. 모델 로드 및 LoRA 구성
import torch
from transformers import AutoModelForCausalLM
from peft import LoraConfig, PeftModel, get_peft_model

use_bf16 = torch.cuda.is_bf16_supported()
model_dtype = torch.bfloat16 if use_bf16 else torch.float16

base_model = AutoModelForCausalLM.from_pretrained(
    str(BASE_MODEL_DIR),
    torch_dtype=model_dtype,
    device_map={"": 0},
)
base_model.config.use_cache = False

if (ADAPTER_DIR / "adapter_config.json").exists() and not FORCE_RETRAIN:
    print("기존 최종 LoRA adapter 로드:", ADAPTER_DIR)
    model = PeftModel.from_pretrained(
        base_model,
        str(ADAPTER_DIR),
        is_trainable=False,
    )
    TRAIN_NEEDED = False
else:
    lora_config = LoraConfig(
        r=16,
        lora_alpha=32,
        lora_dropout=0.05,
        bias="none",
        task_type="CAUSAL_LM",
        target_modules=[
            "q_proj","k_proj","v_proj","o_proj",
            "gate_proj","up_proj","down_proj",
        ],
    )
    model = get_peft_model(base_model, lora_config)
    model.enable_input_require_grads()
    model.print_trainable_parameters()
    TRAIN_NEEDED = True

print("학습 dtype:", model_dtype)
print("학습 필요:", TRAIN_NEEDED)


trainable params: 10,092,544 || all params: 606,142,464 || trainable%: 1.6650
학습 dtype: torch.bfloat16
학습 필요: True


In [ ]:
# 10. Trainer 생성 및 체크포인트 자동 이어서 학습
from transformers import DataCollatorForSeq2Seq, Trainer, TrainingArguments
from transformers.trainer_utils import get_last_checkpoint

data_collator = DataCollatorForSeq2Seq(
    tokenizer=tokenizer,
    padding=True,
    label_pad_token_id=-100,
    return_tensors="pt",
)

trainer = None

if TRAIN_NEEDED:
    training_args = TrainingArguments(
        output_dir=str(CHECKPOINT_DIR),
        overwrite_output_dir=False,
        num_train_epochs=NUM_EPOCHS,
        learning_rate=LEARNING_RATE,
        weight_decay=0.01,
        warmup_ratio=0.1,
        lr_scheduler_type="cosine",
        max_grad_norm=1.0,
        per_device_train_batch_size=TRAIN_BATCH_SIZE,
        per_device_eval_batch_size=EVAL_BATCH_SIZE,
        gradient_accumulation_steps=GRADIENT_ACCUMULATION_STEPS,
        fp16=not use_bf16,
        bf16=use_bf16,
        gradient_checkpointing=True,
        eval_strategy="epoch",
        save_strategy="steps",
        save_steps=SAVE_STEPS,
        save_total_limit=3,
        logging_strategy="steps",
        logging_steps=10,
        report_to="none",
        seed=42,
        data_seed=42,
        group_by_length=True,
        length_column_name="length",
        dataloader_num_workers=0,
        remove_unused_columns=True,
    )

    trainer = Trainer(
        model=model,
        args=training_args,
        train_dataset=tokenized["train"],
        eval_dataset=tokenized["validation"],
        data_collator=data_collator,
    )

    last_checkpoint = get_last_checkpoint(str(CHECKPOINT_DIR))
    print("이어갈 체크포인트:", last_checkpoint)

    train_result = trainer.train(resume_from_checkpoint=last_checkpoint)
    trainer.save_model(str(ADAPTER_DIR))
    tokenizer.save_pretrained(str(ADAPTER_DIR))

    trainer.log_metrics("train", train_result.metrics)
    trainer.save_metrics("train", train_result.metrics)
    trainer.save_state()
    print("최종 LoRA adapter 저장 완료:", ADAPTER_DIR)
else:
    print("최종 LoRA adapter가 이미 있어 학습 건너뜀")


No label_names provided for model class `PeftModelForCausalLM`. Since `PeftModel` hides base models input arguments, if label_names is not given, label_names can't be set automatically within `Trainer`. Note that empty label_names list will be used instead.


이어갈 체크포인트: None


Epoch,Training Loss,Validation Loss
1,0.101500,0.422449
2,0.067600,0.219221
3,0.012200,0.023931
4,0.001600,0.019865


***** train metrics *****
  epoch                    =     4.9557
  total_flos               =  2179351GF
  train_loss               =      0.186
  train_runtime            = 0:32:43.66
  train_samples_per_second =      4.135
  train_steps_per_second   =      0.257
최종 LoRA adapter 저장 완료: /content/drive/MyDrive/qwen3_telegram_pipeline/runs/qwen3_0.6b_telegram_parser_260720/lora_adapter


In [ ]:
# 11. validation loss 평가
from transformers import Trainer, TrainingArguments

model.config.use_cache = False

eval_args = TrainingArguments(
    output_dir=str(EVAL_DIR / "trainer_eval"),
    per_device_eval_batch_size=EVAL_BATCH_SIZE,
    fp16=not use_bf16,
    bf16=use_bf16,
    report_to="none",
    remove_unused_columns=True,
    dataloader_num_workers=0,
)

eval_trainer = Trainer(
    model=model,
    args=eval_args,
    eval_dataset=tokenized["validation"],
    data_collator=data_collator,
)

validation_metrics = eval_trainer.evaluate()
print(json.dumps(validation_metrics, ensure_ascii=False, indent=2))

with (EVAL_DIR / "validation_metrics.json").open("w", encoding="utf-8") as file:
    json.dump(validation_metrics, file, ensure_ascii=False, indent=2)


No label_names provided for model class `PeftModelForCausalLM`. Since `PeftModel` hides base models input arguments, if label_names is not given, label_names can't be set automatically within `Trainer`. Note that empty label_names list will be used instead.


{
  "eval_loss": 0.01953626051545143,
  "eval_model_preparation_time": 0.0315,
  "eval_runtime": 27.7238,
  "eval_samples_per_second": 12.12,
  "eval_steps_per_second": 1.515
}


In [ ]:
# 12. test 325행 액션 정확도 검증
from collections import Counter
from tqdm.auto import tqdm
import time

SYSTEM_PROMPT = (
    "너는 특정 BTCUSDT 리딩 채널의 한국어 메시지를 구조화된 거래 액션 JSON으로 "
    "변환하는 파서다. 메시지에 명시된 행동만 추출하고 추측하지 않는다. "
    "출력은 actions 배열을 가진 JSON 하나만 반환한다."
)

def predict(message: str, max_new_tokens: int = 96) -> str:
    prompt = tokenizer.apply_chat_template(
        [
            {"role": "system", "content": SYSTEM_PROMPT},
            {"role": "user", "content": message},
        ],
        tokenize=False,
        add_generation_prompt=True,
        enable_thinking=False,
    )

    inputs = tokenizer(
        prompt,
        return_tensors="pt",
        add_special_tokens=False,
    ).to(model.device)

    model.config.use_cache = True
    with torch.inference_mode():
        generated = model.generate(
            **inputs,
            max_new_tokens=max_new_tokens,
            do_sample=False,
            num_beams=1,
            use_cache=True,
            pad_token_id=tokenizer.pad_token_id,
            eos_token_id=tokenizer.eos_token_id,
        )
    model.config.use_cache = False

    new_tokens = generated[0, inputs["input_ids"].shape[1]:]
    return tokenizer.decode(new_tokens, skip_special_tokens=True).strip()

def normalize_label(obj):
    if not isinstance(obj, dict) or not isinstance(obj.get("actions", []), list):
        return {"actions": []}
    actions = []
    for action in obj.get("actions", []):
        if isinstance(action, dict):
            actions.append({"type": action.get("type"), "price": action.get("price")})
    return {"actions": actions}

def strict_schema_valid(obj):
    if not isinstance(obj, dict) or set(obj.keys()) != {"actions"}:
        return False
    if not isinstance(obj["actions"], list):
        return False
    for action in obj["actions"]:
        if not isinstance(action, dict) or set(action.keys()) != {"type", "price"}:
            return False
        if action["type"] not in ALLOWED_ACTIONS:
            return False
        if action["price"] is not None and not isinstance(action["price"], (int, float)):
            return False
    return True

OPEN_TYPES = {"OPEN_LONG", "OPEN_SHORT"}
CLOSE_TYPES = {"CLOSE_HALF", "CLOSE_ADDS", "CLOSE_ALL"}

results = []
fatal_errors = Counter()
json_valid = strict_valid = exact_match = action_type_exact = price_exact = 0
latencies = []

model.eval()

for row in tqdm(raw_rows["test"]):
    user_message = row["messages"][1]["content"]
    expected = normalize_label(json.loads(row["messages"][2]["content"]))

    started = time.perf_counter()
    predicted_raw = predict(user_message)
    latencies.append(time.perf_counter() - started)

    result = {
        "message": user_message,
        "expected": expected,
        "predicted_raw": predicted_raw,
        "json_valid": False,
        "strict_schema_valid": False,
        "exact_match": False,
    }

    try:
        predicted_obj = json.loads(predicted_raw)
        predicted = normalize_label(predicted_obj)
        result["json_valid"] = True
        result["strict_schema_valid"] = strict_schema_valid(predicted_obj)
        result["predicted"] = predicted

        json_valid += 1
        strict_valid += int(result["strict_schema_valid"])
        if predicted == expected:
            exact_match += 1
            result["exact_match"] = True

        expected_types = [a["type"] for a in expected["actions"]]
        predicted_types = [a["type"] for a in predicted["actions"]]
        expected_prices = [a["price"] for a in expected["actions"]]
        predicted_prices = [a["price"] for a in predicted["actions"]]

        action_type_exact += int(predicted_types == expected_types)
        price_exact += int(predicted_prices == expected_prices)

        expected_set, predicted_set = set(expected_types), set(predicted_types)
        if "OPEN_LONG" in expected_set and "OPEN_SHORT" in predicted_set:
            fatal_errors["LONG_TO_SHORT"] += 1
        if "OPEN_SHORT" in expected_set and "OPEN_LONG" in predicted_set:
            fatal_errors["SHORT_TO_LONG"] += 1
        if expected_set & OPEN_TYPES and predicted_set & CLOSE_TYPES:
            fatal_errors["OPEN_TO_CLOSE"] += 1
        if expected_set & CLOSE_TYPES and predicted_set & OPEN_TYPES:
            fatal_errors["CLOSE_TO_OPEN"] += 1
        if not expected_types and predicted_types:
            fatal_errors["FALSE_ACTION"] += 1
        if expected_types and not predicted_types:
            fatal_errors["MISSED_ACTION"] += 1
    except Exception as exc:
        result["error"] = str(exc)

    results.append(result)

total = len(results)
metrics = {
    "test_rows": total,
    "json_valid_rate": json_valid / total,
    "strict_schema_valid_rate": strict_valid / total,
    "exact_match_rate": exact_match / total,
    "action_type_exact_rate": action_type_exact / total,
    "price_exact_rate": price_exact / total,
    "fatal_errors": dict(fatal_errors),
    "latency_seconds": {
        "mean": sum(latencies) / len(latencies),
        "min": min(latencies),
        "max": max(latencies),
    },
}

print(json.dumps(metrics, ensure_ascii=False, indent=2))

with (EVAL_DIR / "test_metrics.json").open("w", encoding="utf-8") as file:
    json.dump(metrics, file, ensure_ascii=False, indent=2)

with (EVAL_DIR / "test_predictions.jsonl").open("w", encoding="utf-8") as file:
    for result in results:
        file.write(json.dumps(result, ensure_ascii=False) + "\n")

wrong = [result for result in results if not result["exact_match"]]
with (EVAL_DIR / "wrong_predictions.jsonl").open("w", encoding="utf-8") as file:
    for result in wrong:
        file.write(json.dumps(result, ensure_ascii=False) + "\n")

print("오답 수:", len(wrong))
print("평가 결과 저장:", EVAL_DIR)


  0%|          | 0/325 [00:00<?, ?it/s]

/usr/local/lib/python3.12/dist-packages/transformers/generation/configuration_utils.py:631: UserWarning: `do_sample` is set to `False`. However, `temperature` is set to `0.6` -- this flag is only used in sample-based generation modes. You should set `do_sample=True` or unset `temperature`.
  warnings.warn(
/usr/local/lib/python3.12/dist-packages/transformers/generation/configuration_utils.py:636: UserWarning: `do_sample` is set to `False`. However, `top_p` is set to `0.95` -- this flag is only used in sample-based generation modes. You should set `do_sample=True` or unset `top_p`.
  warnings.warn(
/usr/local/lib/python3.12/dist-packages/transformers/generation/configuration_utils.py:653: UserWarning: `do_sample` is set to `False`. However, `top_k` is set to `20` -- this flag is only used in sample-based generation modes. You should set `do_sample=True` or unset `top_k`.
  warnings.warn(


{
  "test_rows": 325,
  "json_valid_rate": 1.0,
  "strict_schema_valid_rate": 0.9969230769230769,
  "exact_match_rate": 0.9353846153846154,
  "action_type_exact_rate": 0.9384615384615385,
  "price_exact_rate": 0.9415384615384615,
  "fatal_errors": {
    "MISSED_ACTION": 10,
    "FALSE_ACTION": 7
  },
  "latency_seconds": {
    "mean": 1.0507886119199834,
    "min": 0.36096847999988313,
    "max": 2.9746438569995917
  }
}
오답 수: 21
평가 결과 저장: /content/drive/MyDrive/qwen3_telegram_pipeline/runs/qwen3_0.6b_telegram_parser_260720/evaluation


In [ ]:
# 13. 오답 출력
wrong = [result for result in results if not result["exact_match"]]
for index, result in enumerate(wrong, 1):
    print(f"\n[{index}]")
    print("메시지:", result["message"])
    print("정답:", json.dumps(result["expected"], ensure_ascii=False))
    print("예측:", result["predicted_raw"])



[1]
메시지: 조금만 더살게요!
정답: {"actions": [{"type": "ADD", "price": null}]}
예측: {"actions":[]}

[2]
메시지: 쪼금만 더 탈게요 지금!
정답: {"actions": [{"type": "ADD", "price": null}]}
예측: {"actions":[{"type":"CLOSE_ADDS","price":null}]}

[3]
메시지: 위로 열린차트라 물좀 타보고 안되면 본전에 정리하던지 할게요!
정답: {"actions": [{"type": "ADD", "price": null}]}
예측: {"actions":[]}

[4]
메시지: 물 조금씩만 더 탈게요! 314
정답: {"actions": [{"type": "ADD", "price": null}]}
예측: {"actions":[{"type":"ADD","price":314}]}

[5]
메시지: 65500라인 깨져가지고 올라오면 본전이나 약익절에 정리할게요 ! 사인 드리겠습니다 !
정답: {"actions": []}
예측: {"actions":[{"type":"CLOSE_ALL","price":65500}]}

[6]
메시지: 내려오면 물 조금씩만 더 타주세요 750정도 맞추시면됩니다!
정답: {"actions": []}
예측: {"actions":[{"type":"ADD","price":750}]}

[7]
메시지: 296 맞추세여 평단
정답: {"actions": [{"type": "ADD", "price": null}]}
예측: {"actions":[]}

[8]
메시지: 전 정리햇어요 !
정답: {"actions": []}
예측: {"actions":[{"type":"CLOSE_ALL","price":null}]}

[9]
메시지: 지금 양전했는데 최대한 타이트하게 사인드려볼게요 집중들 하고계세요!
정답: {"actions": []}
예측: {"actions":[{"type":"ADD","price":null}]}

[10]
메시지

In [ ]:
# 14. LoRA 병합 모델을 Drive에 저장
import gc
import shutil

if MERGED_DIR.exists():
    shutil.rmtree(MERGED_DIR)
MERGED_DIR.mkdir(parents=True, exist_ok=True)

model.config.use_cache = True
merged_model = model.merge_and_unload()
merged_model.save_pretrained(str(MERGED_DIR), safe_serialization=True)
tokenizer.save_pretrained(str(MERGED_DIR))

try:
    merged_model.generation_config.save_pretrained(str(MERGED_DIR))
except Exception as exc:
    print("generation_config 저장 생략:", exc)

print("병합 모델 저장 완료:", MERGED_DIR)
print(sorted(path.name for path in MERGED_DIR.iterdir()))

del merged_model
del model
if trainer is not None:
    del trainer
del eval_trainer
gc.collect()
torch.cuda.empty_cache()


병합 모델 저장 완료: /content/drive/MyDrive/qwen3_telegram_pipeline/runs/qwen3_0.6b_telegram_parser_260720/merged_model
['added_tokens.json', 'config.json', 'generation_config.json', 'merges.txt', 'model.safetensors', 'special_tokens_map.json', 'tokenizer.json', 'tokenizer_config.json', 'vocab.json']


In [ ]:
# 15. GGUF 변환 전 Qwen3 tokenizer_config 호환 패치
tokenizer_config_path = MERGED_DIR / "tokenizer_config.json"
backup_path = MERGED_DIR / "tokenizer_config.original.json"

with tokenizer_config_path.open("r", encoding="utf-8") as file:
    tokenizer_config = json.load(file)

if not backup_path.exists():
    shutil.copy2(tokenizer_config_path, backup_path)

patched = False
if isinstance(tokenizer_config.get("extra_special_tokens"), list):
    print("extra_special_tokens list 제거:", tokenizer_config["extra_special_tokens"])
    tokenizer_config.pop("extra_special_tokens")
    patched = True

with tokenizer_config_path.open("w", encoding="utf-8") as file:
    json.dump(tokenizer_config, file, ensure_ascii=False, indent=2)

from transformers import AutoTokenizer
conversion_tokenizer = AutoTokenizer.from_pretrained(str(MERGED_DIR), use_fast=True)
print("병합 tokenizer 재로드 성공")
print("패치 적용:", patched)


병합 tokenizer 재로드 성공
패치 적용: False


In [ ]:
quantize_bin.chmod(0o755)
cli_bin.chmod(0o755)

FileNotFoundError: [Errno 2] No such file or directory: '/content/drive/MyDrive/qwen3_telegram_pipeline/tools/llama.cpp/build/bin/llama-cli'

In [ ]:
import os
import shutil
import subprocess
from pathlib import Path

# 임시 컴파일 공간
LOCAL_SRC = Path("/tmp/llama.cpp")
LOCAL_BUILD = LOCAL_SRC / "build"

# 완성 바이너리 영구 저장 위치
DRIVE_BIN = Path(
    "/content/drive/MyDrive/"
    "qwen3_telegram_pipeline/tools/llama_bin"
)

DRIVE_BIN.mkdir(parents=True, exist_ok=True)

# 기존 임시 소스 제거
if LOCAL_SRC.exists():
    shutil.rmtree(LOCAL_SRC)

# 최신 llama.cpp clone
subprocess.run(
    [
        "git",
        "clone",
        "--depth",
        "1",
        "https://github.com/ggml-org/llama.cpp.git",
        str(LOCAL_SRC),
    ],
    check=True,
)

# CMake 설정
configure_result = subprocess.run(
    [
        "cmake",
        "-S", str(LOCAL_SRC),
        "-B", str(LOCAL_BUILD),
        "-DCMAKE_BUILD_TYPE=Release",
        "-DGGML_CUDA=OFF",
        "-DLLAMA_BUILD_TESTS=OFF",
        "-DLLAMA_BUILD_EXAMPLES=ON",
        "-DLLAMA_BUILD_SERVER=OFF",
    ],
    text=True,
    stdout=subprocess.PIPE,
    stderr=subprocess.STDOUT,
)

print(configure_result.stdout)

if configure_result.returncode != 0:
    raise RuntimeError("CMake 설정 실패")

# quantize와 cli만 빌드
build_result = subprocess.run(
    [
        "cmake",
        "--build", str(LOCAL_BUILD),
        "--target",
        "llama-quantize",
        "llama-cli",
        "--parallel", "2",
    ],
    text=True,
    stdout=subprocess.PIPE,
    stderr=subprocess.STDOUT,
)

print(build_result.stdout)

if build_result.returncode != 0:
    raise RuntimeError(
        "llama.cpp 빌드 실패. "
        "위 출력의 마지막 error 부분을 확인하세요."
    )

# 결과 탐색
quantize_src = LOCAL_BUILD / "bin" / "llama-quantize"
cli_src = LOCAL_BUILD / "bin" / "llama-cli"

for path in [quantize_src, cli_src]:
    if not path.exists():
        raise FileNotFoundError(
            f"빌드는 끝났지만 파일이 없습니다: {path}"
        )

# Drive로 복사
quantize_bin = DRIVE_BIN / "llama-quantize"
cli_bin = DRIVE_BIN / "llama-cli"

shutil.copy2(quantize_src, quantize_bin)
shutil.copy2(cli_src, cli_bin)

quantize_bin.chmod(0o755)
cli_bin.chmod(0o755)

print("빌드 및 복사 완료")
print("quantize:", quantize_bin)
print("cli:", cli_bin)

-- The C compiler identification is GNU 11.4.0
-- The CXX compiler identification is GNU 11.4.0
-- Detecting C compiler ABI info
-- Detecting C compiler ABI info - done
-- Check for working C compiler: /usr/bin/cc - skipped
-- Detecting C compile features
-- Detecting C compile features - done
-- Detecting CXX compiler ABI info
-- Detecting CXX compiler ABI info - done
-- Check for working CXX compiler: /usr/bin/c++ - skipped
-- Detecting CXX compile features
-- Detecting CXX compile features - done
CMAKE_BUILD_TYPE=Release
-- Found Git: /usr/bin/git (found version "2.34.1")
-- The ASM compiler identification is GNU
-- Found assembler: /usr/bin/cc
-- Performing Test CMAKE_HAVE_LIBC_PTHREAD
-- Performing Test CMAKE_HAVE_LIBC_PTHREAD - Success
-- Found Threads: TRUE
-- Warning: ccache not found - consider installing it for faster compilation or disable this warning with GGML_CCACHE=OFF
-- CMAKE_SYSTEM_PROCESSOR: x86_64
-- GGML_SYSTEM_ARCH: x86
-- Found OpenMP_C: -fopenmp (found version "

KeyboardInterrupt: 

In [ ]:
from pathlib import Path

LLAMA_BIN = LLAMA_CPP_DIR / "build" / "bin"

quantize_candidates = [
    LLAMA_BIN / "llama-quantize",
    LLAMA_BIN / "llama-quantize-cli",
]

for p in quantize_candidates:
    if p.exists():
        quantize_bin = p
        break
else:
    raise FileNotFoundError(
        f"quantize binary not found. files={list(LLAMA_BIN.iterdir())}"
    )

print("quantize:", quantize_bin)

quantize: /content/drive/MyDrive/qwen3_telegram_pipeline/tools/llama.cpp/build/bin/llama-quantize


In [ ]:
# %cd /content/drive/MyDrive/qwen3_telegram_pipeline/tools/llama.cpp

# !cmake -S . -B build \
# -DGGML_CUDA=OFF \
# -DCMAKE_BUILD_TYPE=Release \
# -DLLAMA_BUILD_TESTS=OFF \
# -DLLAMA_BUILD_EXAMPLES=ON

# !cmake --build build --config Release --target llama-quantize -j2

In [ ]:
# # 16. llama.cpp 최신 소스 준비 및 CPU 빌드
# import subprocess

# if (LLAMA_CPP_DIR / ".git").exists():
#     subprocess.run(
#         ["git", "-C", str(LLAMA_CPP_DIR), "fetch", "--depth", "1", "origin", "master"],
#         check=True,
#     )
#     subprocess.run(
#         ["git", "-C", str(LLAMA_CPP_DIR), "reset", "--hard", "origin/master"],
#         check=True,
#     )
# else:
#     subprocess.run(
#         [
#             "git", "clone", "--depth", "1",
#             "https://github.com/ggml-org/llama.cpp.git",
#             str(LLAMA_CPP_DIR),
#         ],
#         check=True,
#     )

# build_dir = LLAMA_CPP_DIR / "build"

# subprocess.run(
#     [
#         "cmake", "-S", str(LLAMA_CPP_DIR), "-B", str(build_dir),
#         "-DGGML_CUDA=OFF", "-DCMAKE_BUILD_TYPE=Release",
#     ],
#     check=True,
# )
# subprocess.run(
#     ["cmake", "--build", str(build_dir), "--config", "Release", "-j", "2"],
#     check=True,
# )

# quantize_bin = build_dir / "bin" / "llama-quantize"
# cli_bin = build_dir / "bin" / "llama-cli"

# if not quantize_bin.exists() or not cli_bin.exists():
#     raise FileNotFoundError("llama.cpp 빌드 산출물을 찾을 수 없습니다.")

# print("llama.cpp 빌드 완료")


In [ ]:
# 17. 병합 HF 모델 → F16 GGUF 변환
import subprocess
import sys

F16_GGUF = GGUF_DIR / f"{RUN_NAME}-F16.gguf"
convert_script = LLAMA_CPP_DIR / "convert_hf_to_gguf.py"

subprocess.run(
    [
        sys.executable,
        str(convert_script),
        str(MERGED_DIR),
        "--outfile", str(F16_GGUF),
        "--outtype", "f16",
    ],
    cwd=str(LLAMA_CPP_DIR),
    check=True,
)

print("F16 GGUF 생성 완료:", F16_GGUF)
print("크기(MB):", round(F16_GGUF.stat().st_size / 1024 / 1024, 2))


F16 GGUF 생성 완료: /content/drive/MyDrive/qwen3_telegram_pipeline/runs/qwen3_0.6b_telegram_parser_260720/gguf/qwen3_0.6b_telegram_parser_260720-F16.gguf
크기(MB): 1142.68


In [ ]:
# 18. Q4_K_M / Q5_K_M / Q8_0 양자화
QUANTIZED_FILES = {}

build_dir = LLAMA_CPP_DIR / "build"

quantize_bin = build_dir / "bin" / "llama-quantize"
cli_bin = build_dir / "bin" / "llama-cli"

for quant_type in ["Q4_K_M", "Q5_K_M", "Q8_0"]:
    output_path = GGUF_DIR / f"{RUN_NAME}-{quant_type}.gguf"
    subprocess.run(
        [
            str(quantize_bin),
            str(F16_GGUF),
            str(output_path),
            quant_type,
        ],
        check=True,
    )
    QUANTIZED_FILES[quant_type] = output_path
    print(
        quant_type,
        "완료 —",
        output_path,
        f"({output_path.stat().st_size / 1024 / 1024:.2f} MB)",
    )


Q4_K_M 완료 — /content/drive/MyDrive/qwen3_telegram_pipeline/runs/qwen3_0.6b_telegram_parser_260720/gguf/qwen3_0.6b_telegram_parser_260720-Q4_K_M.gguf (378.33 MB)
Q5_K_M 완료 — /content/drive/MyDrive/qwen3_telegram_pipeline/runs/qwen3_0.6b_telegram_parser_260720/gguf/qwen3_0.6b_telegram_parser_260720-Q5_K_M.gguf (423.83 MB)
Q8_0 완료 — /content/drive/MyDrive/qwen3_telegram_pipeline/runs/qwen3_0.6b_telegram_parser_260720/gguf/qwen3_0.6b_telegram_parser_260720-Q8_0.gguf (609.82 MB)


In [ ]:
import subprocess
from pathlib import Path

BUILD_DIR = LLAMA_CPP_DIR / "build"
BIN_DIR = BUILD_DIR / "bin"

subprocess.run(
    [
        "cmake",
        "--build", str(BUILD_DIR),
        "--target", "llama-completion",
        "--parallel", "2",
    ],
    check=True,
)

cli_bin = BIN_DIR / "llama-completion"

if not cli_bin.exists():
    raise FileNotFoundError(cli_bin)

print("사용 실행 파일:", cli_bin)

cli_bin.chmod(0o755)

사용 실행 파일: /content/drive/MyDrive/qwen3_telegram_pipeline/tools/llama.cpp/build/bin/llama-completion


In [ ]:
from pathlib import Path

BIN_DIR = LLAMA_CPP_DIR / "build" / "bin"

cli_candidates = [
    BIN_DIR / "llama",        # 최신 통합 CLI
    BIN_DIR / "llama-cli",    # 구버전 호환
]

for p in cli_candidates:
    if p.exists():
        cli_bin = p
        break
else:
    raise FileNotFoundError(
        f"llama 실행 파일 못 찾음: {list(BIN_DIR.glob('*'))}"
    )

print("사용 실행 파일:", cli_bin)

FileNotFoundError: llama 실행 파일 못 찾음: [PosixPath('/content/drive/MyDrive/qwen3_telegram_pipeline/tools/llama.cpp/build/bin/libggml-base.so.0'), PosixPath('/content/drive/MyDrive/qwen3_telegram_pipeline/tools/llama.cpp/build/bin/libggml-base.so'), PosixPath('/content/drive/MyDrive/qwen3_telegram_pipeline/tools/llama.cpp/build/bin/libggml-base.so.0.16.0'), PosixPath('/content/drive/MyDrive/qwen3_telegram_pipeline/tools/llama.cpp/build/bin/llama-llava-cli'), PosixPath('/content/drive/MyDrive/qwen3_telegram_pipeline/tools/llama.cpp/build/bin/llama-gemma3-cli'), PosixPath('/content/drive/MyDrive/qwen3_telegram_pipeline/tools/llama.cpp/build/bin/llama-minicpmv-cli'), PosixPath('/content/drive/MyDrive/qwen3_telegram_pipeline/tools/llama.cpp/build/bin/llama-qwen2vl-cli'), PosixPath('/content/drive/MyDrive/qwen3_telegram_pipeline/tools/llama.cpp/build/bin/libggml-cpu.so.0'), PosixPath('/content/drive/MyDrive/qwen3_telegram_pipeline/tools/llama.cpp/build/bin/libggml-cpu.so'), PosixPath('/content/

In [ ]:
import subprocess

q5_path = GGUF_DIR / f"{RUN_NAME}-Q5_K_M.gguf"

prompt = (
    "<|im_start|>system\n"
    "너는 특정 BTCUSDT 리딩 채널의 한국어 메시지를 구조화된 거래 액션 JSON으로 "
    "변환하는 파서다. 메시지에 명시된 행동만 추출하고 추측하지 않는다. "
    "출력은 actions 배열을 가진 JSON 하나만 반환한다."
    "<|im_end|>\n"
    "<|im_start|>user\n"
    "숏 ㅂㅈ"
    "<|im_end|>\n"
    "<|im_start|>assistant\n"
    "<think>\n\n</think>\n\n"
)

completed = subprocess.run(
    [
        str(cli_bin),
        "-m", str(q5_path),
        "-p", prompt,
        "-n", "48",
        "--temp", "0",
        "--no-conversation",
        "--no-display-prompt",
        "--simple-io",
    ],
    capture_output=True,
    text=True,
    check=False,
)

print("return code:", completed.returncode)
print("stdout:")
print(completed.stdout.strip())

print("\nstderr 마지막:")
print(completed.stderr[-1000:])

return code: 0
stdout:
{"actions":[{"type":"OPEN_SHORT","price":null}]} [end of text]

stderr 마지막:
.559.161 I sampler chain: logits -> ?penalties -> ?dry -> ?top-n-sigma -> top-k -> ?typical -> top-p -> min-p -> ?xtc -> temp-ext -> dist 
0.09.559.200 I generate: n_ctx = 40960, n_batch = 2048, n_predict = 48, n_keep = 0
0.09.559.202 I 
0.18.530.257 I common_perf_print:    sampling time =      10.27 ms
0.18.530.259 I common_perf_print:    samplers time =       4.80 ms /   101 tokens
0.18.530.267 I common_perf_print:        load time =    8558.31 ms
0.18.530.270 I common_perf_print: prompt eval time =    7318.18 ms /    87 tokens (   84.12 ms per token,    11.89 tokens per second)
0.18.530.272 I common_perf_print:        eval time =    1636.65 ms /    13 runs   (  125.90 ms per token,     7.94 tokens per second)
0.18.530.273 I common_perf_print:       total time =    8974.53 ms /   100 tokens
0.18.530.274 I common_perf_print: unaccounted time =       9.44 ms /   0.1 %      (total - sampli

In [ ]:
# 20. 최종 산출물 확인
def show_tree(root: Path, max_depth: int = 3):
    root = root.resolve()
    print(root)
    for path in sorted(root.rglob("*")):
        relative = path.relative_to(root)
        if len(relative.parts) > max_depth:
            continue
        indent = "  " * len(relative.parts)
        if path.is_file():
            size_mb = path.stat().st_size / 1024 / 1024
            print(f"{indent}- {path.name} ({size_mb:.2f} MB)")
        else:
            print(f"{indent}- {path.name}/")

show_tree(RUN_ROOT, max_depth=3)

print("\n완료")
print("LoRA adapter:", ADAPTER_DIR)
print("병합 HF 모델:", MERGED_DIR)
print("평가 결과:", EVAL_DIR)
print("GGUF:", GGUF_DIR)


/content/drive/MyDrive/qwen3_telegram_pipeline/runs/qwen3_0.6b_telegram_parser_260720
  - checkpoints/
    - all_results.json (0.00 MB)
    - checkpoint-450/
      - README.md (0.00 MB)
      - adapter_config.json (0.00 MB)
      - adapter_model.safetensors (38.55 MB)
      - added_tokens.json (0.00 MB)
      - merges.txt (1.59 MB)
      - optimizer.pt (77.31 MB)
      - rng_state.pth (0.01 MB)
      - scheduler.pt (0.00 MB)
      - special_tokens_map.json (0.00 MB)
      - tokenizer.json (10.89 MB)
      - tokenizer_config.json (0.01 MB)
      - trainer_state.json (0.01 MB)
      - training_args.bin (0.01 MB)
      - vocab.json (2.65 MB)
    - checkpoint-500/
      - README.md (0.00 MB)
      - adapter_config.json (0.00 MB)
      - adapter_model.safetensors (38.55 MB)
      - added_tokens.json (0.00 MB)
      - merges.txt (1.59 MB)
      - optimizer.pt (77.31 MB)
      - rng_state.pth (0.01 MB)
      - scheduler.pt (0.00 MB)
      - special_tokens_map.json (0.00 MB)
      - tokenizer.

In [ ]:
# # 21. GGUF 반복 평가용 llama-cpp-python 설치

# import os
# import subprocess
# import sys

# # T4 CUDA 빌드
# env = os.environ.copy()
# env["CMAKE_ARGS"] = "-DGGML_CUDA=on"
# env["FORCE_CMAKE"] = "1"

# subprocess.run(
#     [
#         sys.executable,
#         "-m",
#         "pip",
#         "install",
#         "-q",
#         "--upgrade",
#         "--force-reinstall",
#         "--no-cache-dir",
#         "llama-cpp-python",
#     ],
#     env=env,
#     check=True,
# )

# print("llama-cpp-python 설치 완료")

In [ ]:
!pip install llama-cpp-python

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 71.6/71.6 MB 12.0 MB/s eta 0:00:00
  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Installing backend dependencies ... done
  Preparing metadata (pyproject.toml) ... done
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 45.5/45.5 kB 2.1 MB/s eta 0:00:00
  Created wheel for llama-cpp-python: filename=llama_cpp_python-0.3.34-py3-none-linux_x86_64.whl size=20583859 sha256=6149ec8940271631d441610d78e48838be1527c8f58ff7a23262ccb4e629406e
  Stored in directory: /root/.cache/pip/wheels/4a/10/e7/0eb9b120f1640844f33562a3964c5b18b67de1d66d3f9530e8
Successfully built llama-cpp-python


In [ ]:
# 22. Q4/Q5/Q8 공통 평가 함수

import gc
import json
import re
import time
from collections import Counter
from pathlib import Path

import torch
from llama_cpp import Llama
from tqdm.auto import tqdm


SYSTEM_PROMPT = (
    "너는 특정 BTCUSDT 리딩 채널의 한국어 메시지를 구조화된 거래 액션 JSON으로 "
    "변환하는 파서다. 메시지에 명시된 행동만 추출하고 추측하지 않는다. "
    "출력은 actions 배열을 가진 JSON 하나만 반환한다."
)

OPEN_TYPES = {"OPEN_LONG", "OPEN_SHORT"}
CLOSE_TYPES = {
    "CLOSE_HALF",
    "CLOSE_ADDS",
    "CLOSE_ALL",
}


def build_prompt(message: str) -> str:
    return (
        "<|im_start|>system\n"
        f"{SYSTEM_PROMPT}"
        "<|im_end|>\n"
        "<|im_start|>user\n"
        f"{message}"
        "<|im_end|>\n"
        "<|im_start|>assistant\n"
        "<think>\n\n</think>\n\n"
    )


def normalize_result(obj):
    if not isinstance(obj, dict):
        return {"actions": []}

    raw_actions = obj.get("actions")

    if not isinstance(raw_actions, list):
        return {"actions": []}

    actions = []

    for action in raw_actions:
        if not isinstance(action, dict):
            continue

        actions.append(
            {
                "type": action.get("type"),
                "price": action.get("price"),
            }
        )

    return {"actions": actions}


def extract_json(raw_text: str):
    cleaned = raw_text.replace("[end of text]", "").strip()

    # 가장 먼저 등장하는 actions JSON 추출
    match = re.search(
        r'\{\s*"actions"\s*:\s*\[.*?\]\s*\}',
        cleaned,
        flags=re.DOTALL,
    )

    if not match:
        raise ValueError(
            f"actions JSON을 찾지 못함: {cleaned!r}"
        )

    return json.loads(match.group(0))


def evaluate_gguf(
    quant_name: str,
    model_path: Path,
    test_rows,
):
    print(f"\n===== {quant_name} 평가 시작 =====")
    print("모델:", model_path)

    if not model_path.exists():
        raise FileNotFoundError(model_path)

    load_started = time.perf_counter()

    llm = Llama(
        model_path=str(model_path),

        # 입력이 매우 짧으므로 512면 충분
        n_ctx=512,

        # T4에 가능한 레이어 전부 올림
        n_gpu_layers=-1,

        n_batch=256,
        seed=42,
        verbose=False,
    )

    model_load_seconds = (
        time.perf_counter() - load_started
    )

    results = []
    latencies = []

    exact_count = 0
    type_count = 0
    price_count = 0
    json_valid_count = 0

    error_counts = Counter()

    for row in tqdm(
        test_rows,
        desc=quant_name,
    ):
        message = row["messages"][1]["content"]

        expected = normalize_result(
            json.loads(
                row["messages"][2]["content"]
            )
        )

        prompt = build_prompt(message)

        started = time.perf_counter()

        response = llm(
            prompt,
            max_tokens=48,
            temperature=0.0,
            top_p=1.0,
            top_k=1,
            repeat_penalty=1.0,
            echo=False,
            stop=[
                "<|im_end|>",
                "<|endoftext|>",
            ],
        )

        latency = time.perf_counter() - started
        latencies.append(latency)

        raw_output = (
            response["choices"][0]["text"].strip()
        )

        result = {
            "message": message,
            "expected": expected,
            "raw_output": raw_output,
            "latency_seconds": latency,
            "json_valid": False,
            "exact_match": False,
        }

        try:
            parsed = extract_json(raw_output)
            predicted = normalize_result(parsed)

            result["json_valid"] = True
            result["predicted"] = predicted

            json_valid_count += 1

            expected_types = [
                action["type"]
                for action in expected["actions"]
            ]
            predicted_types = [
                action["type"]
                for action in predicted["actions"]
            ]

            expected_prices = [
                action["price"]
                for action in expected["actions"]
            ]
            predicted_prices = [
                action["price"]
                for action in predicted["actions"]
            ]

            if predicted == expected:
                exact_count += 1
                result["exact_match"] = True

            if predicted_types == expected_types:
                type_count += 1

            if predicted_prices == expected_prices:
                price_count += 1

            expected_set = set(expected_types)
            predicted_set = set(predicted_types)

            if (
                "OPEN_LONG" in expected_set
                and "OPEN_SHORT" in predicted_set
            ):
                error_counts["LONG_TO_SHORT"] += 1

            if (
                "OPEN_SHORT" in expected_set
                and "OPEN_LONG" in predicted_set
            ):
                error_counts["SHORT_TO_LONG"] += 1

            if (
                expected_set & OPEN_TYPES
                and predicted_set & CLOSE_TYPES
            ):
                error_counts["OPEN_TO_CLOSE"] += 1

            if (
                expected_set & CLOSE_TYPES
                and predicted_set & OPEN_TYPES
            ):
                error_counts["CLOSE_TO_OPEN"] += 1

            if not expected_types and predicted_types:
                error_counts["FALSE_ACTION"] += 1

            if expected_types and not predicted_types:
                error_counts["MISSED_ACTION"] += 1

        except Exception as exc:
            result["parse_error"] = str(exc)
            error_counts["INVALID_JSON"] += 1

        results.append(result)

    total = len(results)

    sorted_latencies = sorted(latencies)

    def percentile(values, percentile_value):
        if not values:
            return None

        index = int(
            (len(values) - 1)
            * percentile_value
        )

        return values[index]

    metrics = {
        "quantization": quant_name,
        "model_path": str(model_path),
        "model_size_mb": (
            model_path.stat().st_size
            / 1024
            / 1024
        ),
        "test_rows": total,
        "model_load_seconds": model_load_seconds,

        "json_valid_rate": (
            json_valid_count / total
        ),
        "exact_match_rate": (
            exact_count / total
        ),
        "action_type_exact_rate": (
            type_count / total
        ),
        "price_exact_rate": (
            price_count / total
        ),

        "latency_seconds": {
            "mean": (
                sum(latencies)
                / len(latencies)
            ),
            "p50": percentile(
                sorted_latencies,
                0.50,
            ),
            "p95": percentile(
                sorted_latencies,
                0.95,
            ),
            "max": max(latencies),
        },

        "errors": dict(error_counts),
    }

    output_dir = (
        EVAL_DIR
        / "gguf_benchmark"
        / quant_name
    )
    output_dir.mkdir(
        parents=True,
        exist_ok=True,
    )

    with (
        output_dir / "metrics.json"
    ).open(
        "w",
        encoding="utf-8",
    ) as file:
        json.dump(
            metrics,
            file,
            ensure_ascii=False,
            indent=2,
        )

    with (
        output_dir / "predictions.jsonl"
    ).open(
        "w",
        encoding="utf-8",
    ) as file:
        for result in results:
            file.write(
                json.dumps(
                    result,
                    ensure_ascii=False,
                )
                + "\n"
            )

    wrong_results = [
        result
        for result in results
        if not result["exact_match"]
    ]

    with (
        output_dir / "wrong_predictions.jsonl"
    ).open(
        "w",
        encoding="utf-8",
    ) as file:
        for result in wrong_results:
            file.write(
                json.dumps(
                    result,
                    ensure_ascii=False,
                )
                + "\n"
            )

    print(
        json.dumps(
            metrics,
            ensure_ascii=False,
            indent=2,
        )
    )

    del llm
    gc.collect()

    if torch.cuda.is_available():
        torch.cuda.empty_cache()

    return metrics

In [ ]:
# 23. 양자화 모델별 10건 사전 테스트

GGUF_MODELS = {
    "Q4_K_M": (
        GGUF_DIR
        / f"{RUN_NAME}-Q4_K_M.gguf"
    ),
    "Q5_K_M": (
        GGUF_DIR
        / f"{RUN_NAME}-Q5_K_M.gguf"
    ),
    "Q8_0": (
        GGUF_DIR
        / f"{RUN_NAME}-Q8_0.gguf"
    ),
}

smoke_metrics = {}

for quant_name, model_path in GGUF_MODELS.items():
    smoke_metrics[quant_name] = evaluate_gguf(
        quant_name=f"{quant_name}_SMOKE",
        model_path=model_path,
        test_rows=raw_rows["test"][:10],
    )

print("10건 테스트 완료")


===== Q4_K_M_SMOKE 평가 시작 =====
모델: /content/drive/MyDrive/qwen3_telegram_pipeline/runs/qwen3_0.6b_telegram_parser_260720/gguf/qwen3_0.6b_telegram_parser_260720-Q4_K_M.gguf


Q4_K_M_SMOKE:   0%|          | 0/10 [00:00<?, ?it/s]

{
  "quantization": "Q4_K_M_SMOKE",
  "model_path": "/content/drive/MyDrive/qwen3_telegram_pipeline/runs/qwen3_0.6b_telegram_parser_260720/gguf/qwen3_0.6b_telegram_parser_260720-Q4_K_M.gguf",
  "model_size_mb": 378.3268737792969,
  "test_rows": 10,
  "model_load_seconds": 3.0821542850007972,
  "json_valid_rate": 1.0,
  "exact_match_rate": 0.9,
  "action_type_exact_rate": 0.9,
  "price_exact_rate": 0.9,
  "latency_seconds": {
    "mean": 3.0016996466999446,
    "p50": 2.283588909000173,
    "p95": 5.187663806000273,
    "max": 8.026750832999824
  },
  "errors": {
    "MISSED_ACTION": 1
  }
}

===== Q5_K_M_SMOKE 평가 시작 =====
모델: /content/drive/MyDrive/qwen3_telegram_pipeline/runs/qwen3_0.6b_telegram_parser_260720/gguf/qwen3_0.6b_telegram_parser_260720-Q5_K_M.gguf


Q5_K_M_SMOKE:   0%|          | 0/10 [00:00<?, ?it/s]

{
  "quantization": "Q5_K_M_SMOKE",
  "model_path": "/content/drive/MyDrive/qwen3_telegram_pipeline/runs/qwen3_0.6b_telegram_parser_260720/gguf/qwen3_0.6b_telegram_parser_260720-Q5_K_M.gguf",
  "model_size_mb": 423.8268737792969,
  "test_rows": 10,
  "model_load_seconds": 1.2268306570003915,
  "json_valid_rate": 1.0,
  "exact_match_rate": 0.9,
  "action_type_exact_rate": 0.9,
  "price_exact_rate": 0.9,
  "latency_seconds": {
    "mean": 1.984504315400136,
    "p50": 1.4870881679999002,
    "p95": 2.5235112070004106,
    "max": 5.157920835000368
  },
  "errors": {
    "MISSED_ACTION": 1
  }
}

===== Q8_0_SMOKE 평가 시작 =====
모델: /content/drive/MyDrive/qwen3_telegram_pipeline/runs/qwen3_0.6b_telegram_parser_260720/gguf/qwen3_0.6b_telegram_parser_260720-Q8_0.gguf


Q8_0_SMOKE:   0%|          | 0/10 [00:00<?, ?it/s]

{
  "quantization": "Q8_0_SMOKE",
  "model_path": "/content/drive/MyDrive/qwen3_telegram_pipeline/runs/qwen3_0.6b_telegram_parser_260720/gguf/qwen3_0.6b_telegram_parser_260720-Q8_0.gguf",
  "model_size_mb": 609.8239440917969,
  "test_rows": 10,
  "model_load_seconds": 2.6458607130007294,
  "json_valid_rate": 1.0,
  "exact_match_rate": 0.9,
  "action_type_exact_rate": 0.9,
  "price_exact_rate": 0.9,
  "latency_seconds": {
    "mean": 1.670391114799895,
    "p50": 1.428413446000377,
    "p95": 2.202259495999897,
    "max": 2.9944302679996326
  },
  "errors": {
    "MISSED_ACTION": 1
  }
}
10건 테스트 완료


In [ ]:
# 24. Q4/Q5/Q8 test 325건 전체 평가

all_metrics = {}

for quant_name, model_path in GGUF_MODELS.items():
    all_metrics[quant_name] = evaluate_gguf(
        quant_name=quant_name,
        model_path=model_path,
        test_rows=raw_rows["test"],
    )

summary_path = (
    EVAL_DIR
    / "gguf_benchmark"
    / "benchmark_summary.json"
)

summary_path.parent.mkdir(
    parents=True,
    exist_ok=True,
)

with summary_path.open(
    "w",
    encoding="utf-8",
) as file:
    json.dump(
        all_metrics,
        file,
        ensure_ascii=False,
        indent=2,
    )

print("전체 평가 완료")
print("요약 저장:", summary_path)


===== Q4_K_M 평가 시작 =====
모델: /content/drive/MyDrive/qwen3_telegram_pipeline/runs/qwen3_0.6b_telegram_parser_260720/gguf/qwen3_0.6b_telegram_parser_260720-Q4_K_M.gguf


Q4_K_M:   0%|          | 0/325 [00:00<?, ?it/s]

{
  "quantization": "Q4_K_M",
  "model_path": "/content/drive/MyDrive/qwen3_telegram_pipeline/runs/qwen3_0.6b_telegram_parser_260720/gguf/qwen3_0.6b_telegram_parser_260720-Q4_K_M.gguf",
  "model_size_mb": 378.3268737792969,
  "test_rows": 325,
  "model_load_seconds": 0.980130766000002,
  "json_valid_rate": 1.0,
  "exact_match_rate": 0.9353846153846154,
  "action_type_exact_rate": 0.9353846153846154,
  "price_exact_rate": 0.9384615384615385,
  "latency_seconds": {
    "mean": 1.2472992466492137,
    "p50": 1.0884925959999237,
    "p95": 2.266717293000511,
    "max": 5.893653914999959
  },
  "errors": {
    "MISSED_ACTION": 14,
    "FALSE_ACTION": 5,
    "OPEN_TO_CLOSE": 1
  }
}

===== Q5_K_M 평가 시작 =====
모델: /content/drive/MyDrive/qwen3_telegram_pipeline/runs/qwen3_0.6b_telegram_parser_260720/gguf/qwen3_0.6b_telegram_parser_260720-Q5_K_M.gguf


Q5_K_M:   0%|          | 0/325 [00:00<?, ?it/s]

{
  "quantization": "Q5_K_M",
  "model_path": "/content/drive/MyDrive/qwen3_telegram_pipeline/runs/qwen3_0.6b_telegram_parser_260720/gguf/qwen3_0.6b_telegram_parser_260720-Q5_K_M.gguf",
  "model_size_mb": 423.8268737792969,
  "test_rows": 325,
  "model_load_seconds": 0.8017974290005441,
  "json_valid_rate": 1.0,
  "exact_match_rate": 0.9292307692307692,
  "action_type_exact_rate": 0.9323076923076923,
  "price_exact_rate": 0.9384615384615385,
  "latency_seconds": {
    "mean": 1.8309345691784844,
    "p50": 1.558510904999821,
    "p95": 3.2441092519993617,
    "max": 5.553850731000239
  },
  "errors": {
    "MISSED_ACTION": 12,
    "FALSE_ACTION": 6,
    "OPEN_TO_CLOSE": 1
  }
}

===== Q8_0 평가 시작 =====
모델: /content/drive/MyDrive/qwen3_telegram_pipeline/runs/qwen3_0.6b_telegram_parser_260720/gguf/qwen3_0.6b_telegram_parser_260720-Q8_0.gguf


Q8_0:   0%|          | 0/325 [00:00<?, ?it/s]

{
  "quantization": "Q8_0",
  "model_path": "/content/drive/MyDrive/qwen3_telegram_pipeline/runs/qwen3_0.6b_telegram_parser_260720/gguf/qwen3_0.6b_telegram_parser_260720-Q8_0.gguf",
  "model_size_mb": 609.8239440917969,
  "test_rows": 325,
  "model_load_seconds": 0.9991639219997523,
  "json_valid_rate": 1.0,
  "exact_match_rate": 0.9323076923076923,
  "action_type_exact_rate": 0.9353846153846154,
  "price_exact_rate": 0.9415384615384615,
  "latency_seconds": {
    "mean": 1.6982234733169361,
    "p50": 1.4856887839996489,
    "p95": 3.0428672260004532,
    "max": 4.9101322209999125
  },
  "errors": {
    "MISSED_ACTION": 10,
    "FALSE_ACTION": 7,
    "OPEN_TO_CLOSE": 1
  }
}
전체 평가 완료
요약 저장: /content/drive/MyDrive/qwen3_telegram_pipeline/runs/qwen3_0.6b_telegram_parser_260720/evaluation/gguf_benchmark/benchmark_summary.json


In [ ]:
# 25. Q4/Q5/Q8 비교표

import pandas as pd

summary_rows = []

for quant_name, metrics in all_metrics.items():
    errors = metrics.get("errors", {})
    latency = metrics["latency_seconds"]

    summary_rows.append(
        {
            "quant": quant_name,
            "size_mb": round(
                metrics["model_size_mb"],
                2,
            ),
            "exact_match": round(
                metrics["exact_match_rate"],
                4,
            ),
            "type_exact": round(
                metrics["action_type_exact_rate"],
                4,
            ),
            "price_exact": round(
                metrics["price_exact_rate"],
                4,
            ),
            "json_valid": round(
                metrics["json_valid_rate"],
                4,
            ),
            "false_action": errors.get(
                "FALSE_ACTION",
                0,
            ),
            "missed_action": errors.get(
                "MISSED_ACTION",
                0,
            ),
            "open_to_close": errors.get(
                "OPEN_TO_CLOSE",
                0,
            ),
            "close_to_open": errors.get(
                "CLOSE_TO_OPEN",
                0,
            ),
            "mean_latency_ms": round(
                latency["mean"] * 1000,
                1,
            ),
            "p95_latency_ms": round(
                latency["p95"] * 1000,
                1,
            ),
            "load_seconds": round(
                metrics["model_load_seconds"],
                2,
            ),
        }
    )

comparison_df = pd.DataFrame(
    summary_rows
).sort_values(
    by=[
        "exact_match",
        "false_action",
        "mean_latency_ms",
    ],
    ascending=[
        False,
        True,
        True,
    ],
)

display(comparison_df)

csv_path = (
    EVAL_DIR
    / "gguf_benchmark"
    / "benchmark_summary.csv"
)

comparison_df.to_csv(
    csv_path,
    index=False,
    encoding="utf-8-sig",
)

print("CSV 저장:", csv_path)

,quant,size_mb,exact_match,type_exact,price_exact,json_valid,false_action,missed_action,open_to_close,close_to_open,mean_latency_ms,p95_latency_ms,load_seconds
0,Q4_K_M,378.33,0.9354,0.9354,0.9385,1.0,5,14,1,0,1247.3,2266.7,0.98
2,Q8_0,609.82,0.9323,0.9354,0.9415,1.0,7,10,1,0,1698.2,3042.9,1.00
1,Q5_K_M,423.83,0.9292,0.9323,0.9385,1.0,6,12,1,0,1830.9,3244.1,0.80


CSV 저장: /content/drive/MyDrive/qwen3_telegram_pipeline/runs/qwen3_0.6b_telegram_parser_260720/evaluation/gguf_benchmark/benchmark_summary.csv


## 실행이 끊겼을 때

노트북을 다시 열어 **위에서부터 재실행**하면 됩니다.

- 베이스 모델은 Drive에 남아 재다운로드하지 않습니다.
- 최종 LoRA adapter가 없고 checkpoint가 있으면 마지막 checkpoint부터 이어서 학습합니다.
- 최종 LoRA adapter가 있으면 학습을 건너뛰고 검증·병합·GGUF 단계로 진행합니다.
- 처음부터 다시 학습하려면 설정 셀에서 `FORCE_RETRAIN = True`로 변경합니다.

최종 결과 위치:

`/content/drive/MyDrive/qwen3_telegram_pipeline/runs/qwen3_0.6b_telegram_parser/`
